## Training script for CRNN using CTC loss

In [9]:
import torch
from torch import nn
from torch.optim.adamw import AdamW
from torch.utils.data import DataLoader
import numpy as np
import os
from pathlib import Path
import sys
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent 
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from multi_char_classifier.model import CRNN
from multi_char_classifier.dataset import OCRDataset, pad_collate
from multi_char_classifier.utils import load_config, get_device, save_checkpoint
from multi_char_classifier.inference import ctc_decode


config = load_config("config.yaml")
device = get_device()

# Initialize the vocabulary list
dataset_dir = PROJECT_ROOT / config['paths']['dataset10']
vocab_list = open(dataset_dir / config['paths']['vocab_path'], 'r', encoding='utf-8').read()
vocab_list = vocab_list.splitlines()
vocab_list = ["-"] + vocab_list
id_to_char = {i: char for i, char in enumerate(vocab_list)}
char_to_id = {char: i for i, char in enumerate(vocab_list)}
num_classes = len(vocab_list)

Using device: cpu


In [10]:
# 1. Initialize model
model = CRNN(
    img_channel=config['model']['img_channel'],
    num_class=num_classes,
    rnn_hidden=config['model']['rnn_hidden']
).to(device)

# 2. Initialize loss function and optimizer
loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = AdamW(model.parameters(), lr=config['training']['learning_rate'])
        
# 3. Create the dataset instances
dataset_train = OCRDataset(
    csv_path=dataset_dir / config['paths']['train_labels_csv'],
    img_dir=dataset_dir / config['paths']['train_img_dir'],
    vocab_dict=char_to_id,
    transform=None 
)

dataset_val = OCRDataset(
    csv_path=dataset_dir / config['paths']['val_labels_csv'],
    img_dir=dataset_dir / config['paths']['val_img_dir'],
    vocab_dict=char_to_id,
    transform=None 
)

# 4. Create the DataLoaders
dataloader_train = DataLoader(
    dataset_train,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    collate_fn=pad_collate, # Handles the dynamic image widths
    num_workers=4 # Speeds up image loading
)

dataloader_val = DataLoader(
    dataset_val,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    collate_fn=pad_collate, # Handles the dynamic image widths
    num_workers=4 # Speeds up image loading
)

In [11]:
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            # 1. Load data on device
            images, targets, target_lengths, original_widths = batch
            images = images.to(device) 
            targets = targets.to(device)
            target_lengths = target_lengths.to(device)

            # 2. Pedict sequence
            log_probs = model(images)

            # 3. Calculate CTC loss
            log_probs = log_probs.permute(1, 0, 2) # (batch, sequence, class) -> (time, sequence, class)
            input_lengths = CRNN.compute_seq_len(original_widths).to(device)
            ctc_loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

            total_loss += ctc_loss

    average_loss = total_loss / len(dataloader)
    print("Average validation loss: ", average_loss)
    return average_loss

In [ ]:
max_epochs = config['training']['max_epochs']
eval_interval = config['training']['eval_interval']
best_val_loss = np.inf

pbar_epochs = tqdm(range(max_epochs), desc="Training Progress")
for epoch in pbar_epochs:
    model.train()
    pbar_batches = tqdm(dataloader_train, desc=f"Epoch {epoch}", leave=False)
    for batch in pbar_batches:
        # 1. Load data on device
        images, targets, target_lengths, original_widths = batch
        images = images.to(device) 
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        # 2. Perform prediction
        log_probs = model(images)

        # 3. Calculate CTC loss
        log_probs = log_probs.permute(1, 0, 2) # (batch, sequence, class) -> (sequence, batch, class)
        input_lengths = CRNN.compute_seq_len(original_widths).to(device)
        ctc_loss = loss_fn(log_probs, targets, input_lengths, target_lengths)

        # 4. Optimize
        optimizer.zero_grad()
        ctc_loss.backward()
        optimizer.step()

        
        # 5. Calculate CER metric
        correct_sequences = 0
        total_sequences = images.size(0)
        with torch.no_grad():
            batch_preds = log_probs.permute(1, 0, 2) # (sequence, batch, class) -> (batch, sequence, class)
            offset = 0
            for i in range(total_sequences):
                # Reconstruct the ground truths from the 1-d concatenated tensor (in pad_collate)
                curr_length = target_lengths[i].item()
                true_indices = targets[offset : offset + curr_length].tolist()
                true_text = "".join([vocab_list[idx] for idx in true_indices])
                
                # Update the offset for the next image in the batch
                offset += curr_length

                # Only ctc_decode the actual length of the 
                valid_seq_len = input_lengths[i].item()
                valid_log_probs = batch_preds[i][:valid_seq_len, :]
                pred_text, _ = ctc_decode(log_probs=valid_log_probs, class_names=vocab_list)
                
                if pred_text == true_text:
                    correct_sequences += 1

        batch_accuracy = correct_sequences / total_sequences

        pbar_batches.set_postfix({"loss": f"{ctc_loss.item():.4f} | Accuracy: {batch_accuracy:.2%}"})
        
    if epoch % eval_interval == 0:
        val_loss = evaluate(model, dataloader_val)
        
        pbar_epochs.set_postfix({"val_loss": f"{val_loss.item():.4f}"})
        # Save checkpoint if validation loss improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint_dir_path = PROJECT_ROOT / config['paths']['save_dir'] / config['model']['model_name']
            os.makedirs(checkpoint_dir_path, exist_ok=True)
            checkpoint_file_path = checkpoint_dir_path / "best_model_10phrases_val.pth"
            save_checkpoint(model, optimizer, epoch, val_loss.item(), checkpoint_file_path, vocab_list, config)

Training Progress:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 0:   0%|          | 0/625 [00:13<?, ?it/s]

Average validation loss:  tensor(0.0181)
Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model_10phrases_val.pth


Epoch 1:   0%|          | 0/625 [00:14<?, ?it/s]

Average validation loss:  tensor(0.0035)
Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model_10phrases_val.pth


...

Average validation loss:  tensor(0.0016)
Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model_10phrases_val.pth


Epoch 3:   0%|          | 0/625 [00:11<?, ?it/s]

Average validation loss:  tensor(0.0009)
Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model_10phrases_val.pth


Epoch 4:   0%|          | 0/625 [00:12<?, ?it/s]

Average validation loss:  tensor(0.0005)
Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model_10phrases_val.pth


In [ ]:
# Save the model
checkpoint_dir_path = PROJECT_ROOT / config['paths']['save_dir'] / config['model']['model_name']
os.makedirs(checkpoint_dir_path, exist_ok=True)
checkpoint_file_path = checkpoint_dir_path / "best_model.pth"
save_checkpoint(model, optimizer, 1, 0.0192, checkpoint_file_path, vocab_list, config)

Checkpoint saved to c:\Dev\hanzi_classifier\classifier\checkpoints\multi_char_classifier\best_model.pth
